In [29]:
import os
import xarray as xr
import pandas as pd

# Set directory
#output_directory = "/Users/lb962/Documents/GitHub/ESL/data/processed_GESLA"

# Get list of all NetCDF files in the directory
#nc_files = [os.path.join(output_directory, f) for f in os.listdir(output_directory) if f.endswith('.nc')]

# Load and concatenate datasets along a new dimension if needed
#ds = xr.open_mfdataset(nc_files, combine='by_coords')  # or combine='nested' with concat_dim='time' if needed
ds = xr.open_dataset("/Users/lb962/Documents/GitHub/ESL/notebooks/1_data_prep/filtered_GESLA_sea_level.nc")
lat = ds['latitude'].values
lon = ds['longitude'].values

# If lat/lon are scalars or arrays, pair them
latlon_pairs = list(zip(lat.flatten(), lon.flatten()))
unique_latlon = list(set(latlon_pairs))  # remove duplicates

In [55]:
import numpy as np
from shapely.geometry import Point
from shapely.ops import unary_union
import cartopy.io.shapereader as shpreader
from shapely.strtree import STRtree
from scipy.spatial import cKDTree
from tqdm import tqdm  # Optional: for progress bar
# === Step 1: Create 0.25° grid over your region of interest ===
lat_grid = np.arange(48, 64.1, 0.25)
lon_grid = np.arange(-15, 11.1, 0.25)
lat_mesh, lon_mesh = np.meshgrid(lat_grid, lon_grid, indexing='ij')
grid_points = np.column_stack([lat_mesh.ravel(), lon_mesh.ravel()])  # shape (N, 2)

# === Step 2: Load land geometries ===
reader = shpreader.natural_earth(resolution='50m', category='physical', name='land')
land_polygons = list(shpreader.Reader(reader).geometries())
land_shapes = list(land_polygons)
tree = STRtree(land_shapes)

# === Step 3: Filter to ocean points ===
def is_land(lon, lat):
    point = Point(lon, lat)
    matches = tree.query(point, predicate='intersects')  # returns indices
    return any(land_shapes[i].contains(point) for i in matches)

print("🔍 Checking which grid points are over ocean...")
ocean_mask = [not is_land(lon, lat) for lat, lon in tqdm(grid_points)]
ocean_points = np.array(grid_points)[ocean_mask]

# === Step 4: KDTree for nearest-neighbor search ===
ocean_tree = cKDTree(ocean_points)

# === Step 5: Snap each original point to nearest ocean grid point ===
input_coords = np.array(list(set(unique_latlon)))  # deduplicate, just in case
distances, indices = ocean_tree.query(input_coords)
snapped_to_ocean = ocean_points[indices]

# Step 7: Convert to set to remove duplicates, then sort as list
snapped_unique = sorted(list(set(map(tuple, snapped_to_ocean))))

#for orig, snap in matched[:5]:
  #  print(f"From {orig} → nearest ocean {snap}")

🔍 Checking which grid points are over ocean...


100%|██████████| 6825/6825 [00:01<00:00, 6160.91it/s]


In [60]:
import os
import cdsapi

output_folder = "/Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location"

client = cdsapi.Client()

for i, (lat, lon) in enumerate(snapped_unique):
    out_file = os.path.join(output_folder, f"ERA5_{lat:.2f}_{lon:.2f}.zip")

    if os.path.exists(out_file):
        print(f"File already exists: {out_file}. Skipping...")
        continue

    dataset = "reanalysis-era5-single-levels-timeseries"
    request = {
        "variable": [
            "2m_dewpoint_temperature",
            "2m_temperature",
            "total_precipitation",
            "mean_sea_level_pressure",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "surface_pressure",
            "sea_surface_temperature",
            "mean_wave_direction",
            "mean_wave_period",
            "significant_height_of_combined_wind_waves_and_swell"
        ],
        "location": {"longitude": lon, "latitude": lat},
        "date": ["1940-01-01/2025-07-30"],
        "data_format": "netcdf"
    }

    print(f"Requesting data for ({lat}, {lon})...")
    client.retrieve(dataset, request).download(out_file)
    print(f"Downloaded to: {out_file}")


2025-08-05 15:30:13,566 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.


File already exists: /Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/ERA5_49.25_-2.00.zip. Skipping...
File already exists: /Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/ERA5_49.50_0.00.zip. Skipping...
File already exists: /Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/ERA5_49.75_-1.75.zip. Skipping...
File already exists: /Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/ERA5_50.00_-6.25.zip. Skipping...
File already exists: /Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/ERA5_50.00_-5.50.zip. Skipping...
File already exists: /Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/ERA5_50.00_1.00.zip. Skipping...
File already exists: /Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/ERA5_50.25_-4.25.zip. Skipping...
File already exists: /Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/ERA5_50.50_-2.75.zip. Skipping...
File already exists: /Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/ERA5_50.50_-2.50.zip. Skipping...
Fil

In [58]:
import cdsapi
output_folder = "/Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location"
for i, (lat, lon) in enumerate(snapped_unique):
    out_file = os.path.join(output_folder, f"ERA5_{lat:.2f}_{lon:.2f}.zip")
    
    dataset = "reanalysis-era5-single-levels-timeseries"
    request = {
        "variable": [
            "2m_dewpoint_temperature",
            "2m_temperature",
            "total_precipitation",
            "mean_sea_level_pressure",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "surface_pressure",
            "sea_surface_temperature",
            "mean_wave_direction",
            "mean_wave_period",
            "significant_height_of_combined_wind_waves_and_swell"
        ],
        "location": {"longitude":  lon, "latitude": lat,},
        "date": ["1940-01-01/2025-07-30"],
        "data_format": "netcdf"
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request).download(out_file)

2025-08-05 09:19:36,463 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-08-05 09:19:36,861 WARNING [2025-03-17T00:00:00] Please be aware that the generation of this dataset is using an alternative source for the ERA5 data and may be subject to changes over time (e.g. file format, data file structure, deprecation etc). This dataset should therefore be regarded as “experimental” and is **not recommended for use in a production environment**. 

Notification of changes via this catalogue entry banner and/or in the [Forum](https://forum.ecmwf.int/) will be provided on best efforts.
2025-08-05 09:19:36,862 INFO Request ID is 35692e7a-a014-4ffa-ab3d-a2b70c2d8a9c
2025-08-05 09:19:37,022 INFO status has been updated to accepted
2025-08-05 09:19:42,463 INFO status has been updated to running
2025-08-05 09:19:51,341 INFO status has been updated to successful
2025-08-05 09:20:00,151 INFO [2024-09-26T00:00:00] Watch our

KeyboardInterrupt: 